# 🧱 Lab: Basic RAG Pipeline

**Module 6: Building RAG Systems** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a complete RAG pipeline from document ingestion to answer generation
2. **Implement** chunking strategies to split documents into retrievable segments
3. **Create** a ChromaDB vector store and populate it with embedded documents
4. **Apply** context injection to generate grounded answers from retrieved chunks

## Concepts Covered

| Concept | From |
|---------|------|
| Text Embeddings | mini-embeddings |
| Cosine Similarity | mini-embeddings |
| Text Chunking | mini-chunking |
| Vector Database | mini-vector-db |
| Context Retrieval | This lab |
| Context Injection | This lab |
| Answer Generation | This lab |

## Prerequisites

- **mini-embeddings**: Text embeddings and cosine similarity
- **mini-vector-db**: Vector database operations with ChromaDB
- **mini-chunking**: Document chunking strategies

In [3]:
import json
import numpy as np
from pathlib import Path
from openai import OpenAI
import chromadb
from dotenv import load_dotenv
from IPython.display import Markdown, display

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))
    
load_dotenv()
client = OpenAI()

# Load corpus
corpus_path = Path('../data/ai_engineering_docs')
documents = []
for file in corpus_path.glob('*.json'):
    with open(file, encoding='utf-8') as f:
        docs = json.load(f)
        documents.extend(docs)

print(f'Loaded {len(documents)} documents from corpus')

Loaded 70 documents from corpus


## 2. Text Embeddings

In [4]:
def get_embedding(text, model='text-embedding-3-small'):
    '''Get embedding vector for text.'''
    response = client.embeddings.create(model=model, input=text)
    return response.data[0].embedding

def cosine_similarity(a, b):
    '''Compute cosine similarity between two vectors.'''
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Test embeddings
texts = [
    'How does self-attention work in transformers?',
    'What is the attention mechanism in neural networks?',
    'How to make a chocolate cake?'
]

embeddings = [get_embedding(t) for t in texts]

print('Similarity between similar texts:', 
      round(cosine_similarity(embeddings[0], embeddings[1]), 3))
print('Similarity between different texts:', 
      round(cosine_similarity(embeddings[0], embeddings[2]), 3))

Similarity between similar texts: 0.561
Similarity between different texts: 0.034


## 3. Chunking Strategies

In [5]:
def fixed_size_chunks(text, chunk_size=500, overlap=50):
    '''Split text into fixed-size chunks with overlap.'''
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

def recursive_split(text, chunk_size=500, separators=None):
    '''Recursively split text by separators.'''
    separators = separators or ['\n\n', '\n', '. ', ' ']
    
    if len(text) <= chunk_size:
        return [text]
    
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ''
            for part in parts:
                if len(current) + len(part) < chunk_size:
                    current += part + sep
                else:
                    if current:
                        chunks.append(current.strip())
                    current = part + sep
            if current:
                chunks.append(current.strip())
            return chunks
    
    return fixed_size_chunks(text, chunk_size)

# Test chunking
sample_doc = documents[0]['content']
chunks = recursive_split(sample_doc, chunk_size=300)
print(f'Document split into {len(chunks)} chunks')
print(f'First chunk ({len(chunks[0])} chars):')
print(chunks[0][:200] + '...')

Document split into 4 chunks
First chunk (103 chars):
# openai.chat.completions.create()

Creates a chat completion for the provided messages.

## Parameters...


## 4. ChromaDB Vector Store

In [6]:
# Initialize ChromaDB
chroma_client = chromadb.Client()

# Delete collection if exists
try:
    chroma_client.delete_collection('ai_docs')
except:
    pass

collection = chroma_client.create_collection(
    name='ai_docs',
    metadata={'description': 'AI Engineering documentation'}
)

# Process and add documents
all_chunks = []
all_ids = []
all_metadata = []

for doc in documents[:20]:  # Limit for demo
    chunks = recursive_split(doc['content'], chunk_size=400)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{doc['id']}_chunk_{i}")
        all_metadata.append({
            'source': doc['id'],
            'title': doc['title'],
            'type': doc['type']
        })

# Add to collection (ChromaDB will embed automatically)
collection.add(
    documents=all_chunks,
    ids=all_ids,
    metadatas=all_metadata
)

print(f'Added {len(all_chunks)} chunks to vector store')

Added 74 chunks to vector store


## 5. Building the Retriever

In [7]:
def retrieve(query, n_results=3):
    '''Retrieve relevant chunks for a query.'''
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    return results

# Test retrieval
query = 'How does self-attention work?'
results = retrieve(query, n_results=3)

print(f'Query: {query}\n')
print('Retrieved chunks:')
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], 
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f'\n{i+1}. [{meta["title"]}] (distance: {dist:.3f})')
    print(f'   {doc[:150]}...')

Query: How does self-attention work?

Retrieved chunks:

1. [Self-Attention Mechanism] (distance: 0.698)
   # Self-Attention Mechanism

## Definition

Self-attention, also known as intra-attention, is a mechanism that allows each position in a sequence to at...

2. [Self-Attention Mechanism] (distance: 0.919)
   Think of attention as a soft dictionary lookup:
- Query: "What am I looking for?"
- Keys: "What do I contain?"
- Values: "What information do I provid...

3. [Residual Connections] (distance: 1.113)
   ```python
# Attention block with residual
x = x + self.attention(self.norm1(x))

# Feed-forward block with residual  
x = x + self.feed_forward(self.n...


## 6. Context Injection

In [8]:
def build_context(results, max_tokens=2000):
    '''Build context string from retrieved results.'''
    context_parts = []
    total_chars = 0
    
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        if total_chars + len(doc) > max_tokens * 4:  # Rough char-to-token estimate
            break
        context_parts.append(f"[Source: {meta['title']}]\n{doc}")
        total_chars += len(doc)
    
    return '\n\n---\n\n'.join(context_parts)

# Build context for query
context = build_context(results)
print(f'Context length: {len(context)} characters')

Context length: 903 characters


## 7. Complete RAG Pipeline

In [9]:
def rag_query(question, n_results=3):
    '''Complete RAG pipeline: retrieve, build context, generate answer.'''
    
    # Step 1: Retrieve relevant chunks
    results = retrieve(question, n_results=n_results)
    
    # Step 2: Build context
    context = build_context(results)
    
    # Step 3: Generate answer
    prompt = f'''Answer the question based on the following context.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:'''
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.3
    )
    
    answer = response.choices[0].message.content
    sources = [m['title'] for m in results['metadatas'][0]]
    
    return {'answer': answer, 'sources': sources}

# Test the complete pipeline
questions = [
    'What is self-attention?',
    'How do I handle rate limits with the OpenAI API?',
    'What is prompt injection?'
]

for q in questions:
    print(f'\nQ: {q}')
    result = rag_query(q)
    print(f'A: {result["answer"][:300]}...')
    print(f'Sources: {result["sources"]}')


Q: What is self-attention?
A: Self-attention, also known as intra-attention, is a mechanism that allows each position in a sequence to attend to all positions in the same sequence. It computes representations by relating different positions of a single sequence....
Sources: ['Self-Attention Mechanism', 'Self-Attention Mechanism', 'Residual Connections']

Q: How do I handle rate limits with the OpenAI API?
A: I don't have enough information....
Sources: ['asyncio with LLM APIs', 'openai.embeddings.create()', 'tiktoken.encoding_for_model()']

Q: What is prompt injection?
A: I don't have enough information....
Sources: ['Self-Attention Mechanism', 'anthropic.messages.create()', 'openai.chat.completions.create()']


## 🎯 Summary

### What You Built

You built a complete RAG pipeline that embeds and chunks documents, stores them in a ChromaDB vector store, retrieves relevant context for user queries, and generates grounded answers with source attribution.

### Key Takeaways

1. **Text embeddings** — convert text to numerical vectors that capture semantic meaning, enabling similarity-based retrieval
2. **Chunking** — splitting documents into smaller pieces improves retrieval precision and fits within context windows
3. **Vector store** — ChromaDB provides efficient similarity search over embedded document chunks
4. **Context injection** — retrieved chunks are injected into prompts to ground LLM answers in actual documents
5. **End-to-end RAG** — the retrieve-then-generate pattern combines search relevance with LLM fluency